In [ ]:
import os
import json
import base64
import requests
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic import hub
from langchain_core.tools import tool

load_dotenv()

# ── LLM ──
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

# ── Research Tools ──
web_search = DuckDuckGoSearchRun()
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1000)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)


# ── Mermaid Tool ──
@tool
def create_mermaid_diagram(mermaid_code: str) -> str:
    """Render a Mermaid diagram to a PNG file and return the file path.
    Input must be valid Mermaid syntax (e.g. graph TD, flowchart LR, etc.).
    Do NOT wrap the code in markdown fences — pass raw Mermaid code only."""
    os.makedirs("output/diagrams", exist_ok=True)
    encoded = base64.urlsafe_b64encode(mermaid_code.encode("utf-8")).decode("ascii")
    url = f"https://mermaid.ink/img/{encoded}?type=png&bgColor=white"
    resp = requests.get(url, timeout=30)
    if resp.status_code != 200:
        return f"Error rendering diagram: HTTP {resp.status_code}. Check your Mermaid syntax."
    existing = [f for f in os.listdir("output/diagrams") if f.startswith("diagram_")]
    idx = len(existing) + 1
    path = f"output/diagrams/diagram_{idx}.png"
    with open(path, "wb") as f:
        f.write(resp.content)
    return f"Diagram saved to {path}"


# ── Save Section Tool ──
@tool
def save_research_section(input_str: str) -> str:
    """Save a research section to be included in the final PDF report.
    Input MUST be a JSON string with "title" and "content" keys.
    Example: {"title": "My Title", "content": "My detailed findings..."}"""
    os.makedirs("output", exist_ok=True)
    # Strip outer quotes that ReAct agents sometimes add
    raw = input_str.strip()
    if (raw.startswith("'") and raw.endswith("'")) or (raw.startswith('"') and raw.endswith('"')):
        raw = raw[1:-1]
    try:
        data = json.loads(raw)
        title = data["title"]
        content = data["content"]
    except (json.JSONDecodeError, KeyError):
        # Second attempt: the agent may have double-escaped
        try:
            data = json.loads(json.loads(f'"{raw}"'))
            title = data["title"]
            content = data["content"]
        except Exception:
            lines = input_str.strip().split("\n", 1)
            title = lines[0].strip()[:100]
            content = lines[1].strip() if len(lines) > 1 else input_str.strip()
    sections_file = "output/sections.json"
    sections = []
    if os.path.exists(sections_file):
        with open(sections_file) as f:
            sections = json.load(f)
    sections.append({"title": title, "content": content})
    with open(sections_file, "w") as f:
        json.dump(sections, f)
    return f"Section '{title}' saved ({len(content)} chars). Total sections: {len(sections)}"


tools = [web_search, wiki_tool, create_mermaid_diagram, save_research_section]
print("Tools loaded:", [t.name for t in tools])

In [8]:
# ── Clean previous output ──
import shutil
if os.path.exists("output"):
    shutil.rmtree("output")
os.makedirs("output/diagrams", exist_ok=True)

# ── Agent Setup ──
prompt_template = hub.pull("hwchase17/react")
agent = create_react_agent(llm, tools, prompt_template)
agent_executor = AgentExecutor(
    agent=agent, tools=tools,
    verbose=True, handle_parsing_errors=True,
    max_iterations=25,
)

# ── User Goal ──
user_goal = """Research the evolution of AI agents and agentic architectures.

Your task — do ALL of the following steps:

1. RESEARCH: Use web search and Wikipedia to gather info on:
   - What AI agents are (definition, components: LLM, tools, memory, planning)
   - Key architectures: ReAct, Chain-of-Thought, Tool-Use, Multi-Agent systems
   - Real-world applications (coding assistants, research agents, autonomous systems)
   - Future trends and challenges

2. SAVE SECTIONS: After each research area, use save_research_section to save your findings with a clear title and detailed content.

3. CREATE DIAGRAMS: Create Mermaid diagrams to visualize:
   - A flowchart showing the core components of an AI agent (LLM + Tools + Memory + Planning)
   - A flowchart showing the ReAct loop (Thought → Action → Observation → repeat)
   - A diagram comparing single-agent vs multi-agent architectures
   Use create_mermaid_diagram with raw Mermaid syntax (no markdown fences).

Complete ALL steps before giving your final answer."""

print(f"User Goal:\n{user_goal}\n")
print("-" * 60)
print("Initiating Agent Execution...\n")

response = agent_executor.invoke({"input": user_goal})

print("\n" + "-" * 60)
print(f"Final Output:\n{response['output']}")

User Goal:
Research the evolution of AI agents and agentic architectures.

Your task — do ALL of the following steps:

1. RESEARCH: Use web search and Wikipedia to gather info on:
   - What AI agents are (definition, components: LLM, tools, memory, planning)
   - Key architectures: ReAct, Chain-of-Thought, Tool-Use, Multi-Agent systems
   - Real-world applications (coding assistants, research agents, autonomous systems)
   - Future trends and challenges

2. SAVE SECTIONS: After each research area, use save_research_section to save your findings with a clear title and detailed content.

3. CREATE DIAGRAMS: Create Mermaid diagrams to visualize:
   - A flowchart showing the core components of an AI agent (LLM + Tools + Memory + Planning)
   - A flowchart showing the ReAct loop (Thought → Action → Observation → repeat)
   - A diagram comparing single-agent vs multi-agent architectures
   Use create_mermaid_diagram with raw Mermaid syntax (no markdown fences).

Complete ALL steps before giv

ValidationError: 1 validation error for SaveSectionInput
content
  Field required [type=missing, input_value={'title': '{"title": "Def...ynamic environments."}'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

In [ ]:
# ── Compile PDF Report ──
import re
from fpdf import FPDF
from glob import glob
from PIL import Image

PAGE_W = 210  # A4 width in mm
PAGE_H = 297  # A4 height in mm
MARGIN_TOP = 22  # header + gap
MARGIN_BOT = 20
USABLE_H = PAGE_H - MARGIN_TOP - MARGIN_BOT  # ~255mm
MAX_IMG_W = 180


class ResearchPDF(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 10)
        self.set_text_color(100, 100, 100)
        self.cell(0, 8, "AI Agents Research Report", align="C", new_x="LMARGIN", new_y="NEXT")
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(4)

    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.set_text_color(128, 128, 128)
        self.cell(0, 10, f"Page {self.page_no()}/{{nb}}", align="C")

    def write_markdown(self, text, size=11):
        """Render text with **bold** markdown support."""
        self.set_font("Helvetica", "", size)
        self.set_text_color(50, 50, 50)
        parts = re.split(r"(\*\*.*?\*\*)", text)
        for part in parts:
            if part.startswith("**") and part.endswith("**"):
                self.set_font("Helvetica", "B", size)
                self.write(6, part[2:-2])
                self.set_font("Helvetica", "", size)
            else:
                self.write(6, part)

    def add_image_fit(self, img_path, label):
        """Add an image that fits within the page, scaling down if needed."""
        with Image.open(img_path) as img:
            w_px, h_px = img.size

        # Calculate dimensions — fit to max width first
        img_w = MAX_IMG_W
        img_h = (img_w / w_px) * h_px

        # If still taller than usable area (minus label space), scale down to fit height
        max_img_h = USABLE_H - 24  # reserve space for label + gaps
        if img_h > max_img_h:
            img_h = max_img_h
            img_w = (img_h / h_px) * w_px

        space_needed = 14 + 4 + img_h + 10
        available = PAGE_H - MARGIN_BOT - self.get_y()

        if space_needed > available:
            self.add_page()

        self.set_font("Helvetica", "B", 13)
        self.set_text_color(30, 30, 30)
        self.cell(0, 10, label, new_x="LMARGIN", new_y="NEXT")
        self.ln(2)
        # Center the image horizontally
        x = (PAGE_W - img_w) / 2
        self.image(img_path, x=x, w=img_w, h=img_h)
        self.ln(img_h + 10)


pdf = ResearchPDF()
pdf.alias_nb_pages()
pdf.set_auto_page_break(auto=True, margin=MARGIN_BOT)

# ── Title Page ──
pdf.add_page()
pdf.ln(60)
pdf.set_font("Helvetica", "B", 28)
pdf.set_text_color(30, 30, 30)
pdf.cell(0, 15, "AI Agents & Agentic", align="C", new_x="LMARGIN", new_y="NEXT")
pdf.cell(0, 15, "Architectures", align="C", new_x="LMARGIN", new_y="NEXT")
pdf.ln(10)
pdf.set_font("Helvetica", "", 14)
pdf.set_text_color(80, 80, 80)
pdf.cell(0, 10, "A Visual Research Report", align="C", new_x="LMARGIN", new_y="NEXT")
pdf.ln(5)
pdf.set_font("Helvetica", "I", 11)
pdf.cell(0, 10, "Generated by an AI Agent using LangChain + Mermaid", align="C", new_x="LMARGIN", new_y="NEXT")

# ── Research Sections ──
sections = []
if os.path.exists("output/sections.json"):
    with open("output/sections.json") as f:
        sections = json.load(f)

for i, section in enumerate(sections):
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 18)
    pdf.set_text_color(30, 30, 30)
    pdf.cell(0, 12, f"{i+1}. {section['title']}", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(6)
    pdf.write_markdown(section["content"])
    pdf.ln(8)

# ── Diagrams Section ──
diagram_files = sorted(glob("output/diagrams/diagram_*.png"))
if diagram_files:
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 22)
    pdf.set_text_color(30, 30, 30)
    pdf.cell(0, 14, "Visual Diagrams", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(6)

    diagram_labels = [
        "Core Components of an AI Agent",
        "The ReAct Agent Loop",
        "Single-Agent vs Multi-Agent Architectures",
    ]
    for idx, img_path in enumerate(diagram_files):
        label = diagram_labels[idx] if idx < len(diagram_labels) else f"Diagram {idx + 1}"
        pdf.add_image_fit(img_path, label)

# ── Save ──
output_path = "output/AI_Agents_Research_Report.pdf"
pdf.output(output_path)
print(f"PDF saved to: {output_path}")
print(f"  - {len(sections)} research sections")
print(f"  - {len(diagram_files)} diagrams")